# Experiment 5: Diffusion-Based Virtual Home Staging

**Full pipeline:** Segment → Place → Generate — end-to-end virtual home staging.

**Three stages:**
1. **Mask2Former** (ADE20K) segments the room to find floor/wall regions
2. **Placement heuristics** compute where furniture should go (no manual mask needed)
3. **Paint-by-Example** (fine-tuned) inpaints realistic furniture at the chosen location

**Diffusion approaches evaluated:**
- **Paint-by-Example** — exemplar-guided inpainting (closest to our task)
- **SD Inpainting + IP-Adapter** — image-prompt-conditioned inpainting
- **Paint-by-Example (fine-tuned)** — adapted to our 3D-FRONT dataset

**Advantages over GAN approach:**
- Pre-trained models already understand rooms, furniture, lighting, perspective
- Background preservation is guaranteed (only masked region changes)
- The diffusion model handles spatial generation (not just harmonization)
- No adversarial training instability
- Works well with small datasets (~3000 images)

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate safetensors
!pip install -q pytorch-fid pillow pandas matplotlib tqdm torchmetrics

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
from torchvision import transforms
from torchmetrics.image import StructuralSimilarityIndexMeasure, PeakSignalNoiseRatio

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

POSSIBLE_ROOTS = [
    '/content/drive/MyDrive/processed-256',
    '/content/drive/MyDrive/693-project/data/processed-256',
    '/content/drive/MyDrive/693-project/processed-256',
    '/content/drive/MyDrive/693 project/processed-256',
]

DATA_ROOT = None
for path in POSSIBLE_ROOTS:
    if os.path.isfile(os.path.join(path, 'metadata.csv')):
        DATA_ROOT = path
        break

if DATA_ROOT is None:
    print('Dataset not found. Add the shared folder as a Drive shortcut first.')
    raise FileNotFoundError('See instructions in cell above')

CSV_PATH = os.path.join(DATA_ROOT, 'metadata.csv')
print(f'Dataset root: {DATA_ROOT}')

SAMPLE_DIR = '/content/drive/MyDrive/693-project/samples/diffusion'
os.makedirs(SAMPLE_DIR, exist_ok=True)

In [ ]:
import shutil, time

LOCAL_DATA = '/content/local_data'
if not os.path.isdir(LOCAL_DATA):
    print('Copying dataset to local SSD...')
    t0 = time.time()
    shutil.copytree(DATA_ROOT, LOCAL_DATA)
    print(f'Done in {time.time() - t0:.0f}s')
else:
    print(f'Local copy exists at {LOCAL_DATA}')

DATA_ROOT = LOCAL_DATA
CSV_PATH = os.path.join(DATA_ROOT, 'metadata.csv')

for split in ['train', 'val', 'test']:
    n = len(os.listdir(os.path.join(DATA_ROOT, split, 'input')))
    print(f'  {split}/input: {n} files (local SSD)')

In [ ]:
IMG_SIZE = 512

df = pd.read_csv(CSV_PATH)

def rebase_path(p):
    parts = p.replace('\\', '/').split('/')
    for i, part in enumerate(parts):
        if part in ('train', 'val', 'test'):
            return os.path.join(DATA_ROOT, *parts[i:])
    return p

for col in ['input_path', 'target_path', 'furniture_path']:
    df[col] = df[col].apply(rebase_path)

sample_img = Image.open(df['target_path'].iloc[0])
stored_w, stored_h = sample_img.size
bbox_max = max(df['bbox_x2'].max(), df['bbox_y2'].max())

if bbox_max > max(stored_w, stored_h):
    orig_w = int(np.ceil(df['bbox_x2'].max() / 256) * 256)
    orig_h = int(np.ceil(df['bbox_y2'].max() / 256) * 256)
    for c in ['bbox_x1', 'bbox_x2']:
        df[c] *= stored_w / orig_w
    for c in ['bbox_y1', 'bbox_y2']:
        df[c] *= stored_h / orig_h
    print(f'Rescaled bboxes from {orig_w}x{orig_h} to {stored_w}x{stored_h}')

train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df   = df[df['split'] == 'val'].reset_index(drop=True)
test_df  = df[df['split'] == 'test'].reset_index(drop=True)
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

In [ ]:
def make_bbox_mask(row, img_size=IMG_SIZE, stored_size=256, margin=10):
    """Create a binary mask from bounding box coordinates, scaled to img_size."""
    scale = img_size / stored_size
    x1 = max(0, int(row['bbox_x1'] * scale) - margin)
    y1 = max(0, int(row['bbox_y1'] * scale) - margin)
    x2 = min(img_size, int(row['bbox_x2'] * scale) + margin)
    y2 = min(img_size, int(row['bbox_y2'] * scale) + margin)
    mask = Image.new('L', (img_size, img_size), 0)
    from PIL import ImageDraw
    ImageDraw.Draw(mask).rectangle([x1, y1, x2, y2], fill=255)
    return mask


def load_and_resize(path, size=IMG_SIZE):
    return Image.open(path).convert('RGB').resize((size, size), Image.BILINEAR)


# Visualize a few samples with their masks
fig, axes = plt.subplots(4, 4, figsize=(16, 16))
for i in range(4):
    row = test_df.iloc[i]
    room = load_and_resize(row['input_path'])
    target = load_and_resize(row['target_path'])
    furn = load_and_resize(row['furniture_path'])
    mask = make_bbox_mask(row)

    axes[i, 0].imshow(room); axes[i, 0].set_title('Empty Room'); axes[i, 0].axis('off')
    axes[i, 1].imshow(mask, cmap='gray'); axes[i, 1].set_title('BBox Mask'); axes[i, 1].axis('off')
    axes[i, 2].imshow(furn); axes[i, 2].set_title('Furniture Ref'); axes[i, 2].axis('off')
    axes[i, 3].imshow(target); axes[i, 3].set_title('Ground Truth'); axes[i, 3].axis('off')
plt.suptitle('Data Samples with BBox Masks', fontsize=16)
plt.tight_layout()
plt.show()

## Approach 1: Paint-by-Example (Zero-Shot)

Paint-by-Example is an exemplar-guided inpainting model built on Stable Diffusion 1.4.  
It takes an image, a mask, and a reference image, and inpaints the masked region to look like the reference.

This is the closest pre-trained model to our task -- no training required for initial evaluation.

In [ ]:
from diffusers import PaintByExamplePipeline

pipe_pbe = PaintByExamplePipeline.from_pretrained(
    'Fantasy-Studio/Paint-by-Example',
    torch_dtype=torch.float16,
    safety_checker=None,
)
pipe_pbe = pipe_pbe.to('cuda')
pipe_pbe.set_progress_bar_config(disable=True)
print('Paint-by-Example pipeline loaded.')

In [ ]:
NUM_STEPS = 50
GUIDANCE_SCALE = 5.0


@torch.inference_mode()
def run_paint_by_example(pipe, split_df, num_samples=None, seed=42):
    """Run Paint-by-Example on a split and return generated images + metrics."""
    generator = torch.Generator('cuda').manual_seed(seed)
    results = []
    n = len(split_df) if num_samples is None else min(num_samples, len(split_df))

    for i in tqdm(range(n), desc='Paint-by-Example'):
        row = split_df.iloc[i]
        room = load_and_resize(row['input_path'])
        target = load_and_resize(row['target_path'])
        furn = load_and_resize(row['furniture_path'])
        mask = make_bbox_mask(row)

        output = pipe(
            image=room,
            mask_image=mask,
            example_image=furn,
            num_inference_steps=NUM_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            generator=generator,
        ).images[0]

        results.append({
            'room': room,
            'furniture': furn,
            'generated': output,
            'target': target,
            'mask': mask,
        })

    return results


# Run on first 8 test samples to preview
pbe_preview = run_paint_by_example(pipe_pbe, test_df, num_samples=8)

In [ ]:
def show_results(results, title, n=None):
    n = min(n or len(results), len(results), 8)
    fig, axes = plt.subplots(n, 4, figsize=(20, 5 * n))
    col_titles = ['Empty Room', 'Furniture Ref', 'Generated', 'Ground Truth']
    for i in range(n):
        r = results[i]
        axes[i, 0].imshow(r['room'])
        axes[i, 1].imshow(r['furniture'])
        axes[i, 2].imshow(r['generated'])
        axes[i, 3].imshow(r['target'])
        for j in range(4):
            axes[i, j].axis('off')
            if i == 0:
                axes[i, j].set_title(col_titles[j], fontsize=14, fontweight='bold')
    plt.suptitle(title, fontsize=18, fontweight='bold', y=1.005)
    plt.tight_layout()
    plt.savefig(os.path.join(SAMPLE_DIR, f'{title.lower().replace(" ", "_")}.png'),
                dpi=150, bbox_inches='tight')
    plt.show()


show_results(pbe_preview, 'Paint-by-Example Zero-Shot')

In [ ]:
def compute_metrics(results, eval_size=256):
    """Compute SSIM and PSNR between generated and target images."""
    to_tensor = transforms.Compose([
        transforms.Resize((eval_size, eval_size)),
        transforms.ToTensor(),
    ])
    ssim_vals, psnr_vals = [], []

    for r in results:
        gen_t = to_tensor(r['generated']).unsqueeze(0)
        tgt_t = to_tensor(r['target']).unsqueeze(0)
        ssim_vals.append(StructuralSimilarityIndexMeasure(data_range=1.0)(gen_t, tgt_t).item())
        psnr_vals.append(PeakSignalNoiseRatio(data_range=1.0)(gen_t, tgt_t).item())

    return {
        'ssim_mean': float(np.mean(ssim_vals)),
        'ssim_std':  float(np.std(ssim_vals)),
        'psnr_mean': float(np.mean(psnr_vals)),
        'psnr_std':  float(np.std(psnr_vals)),
        'ssim_all':  ssim_vals,
        'psnr_all':  psnr_vals,
    }


# Run on full test set
print('Running Paint-by-Example on full test set...')
pbe_results = run_paint_by_example(pipe_pbe, test_df)
pbe_metrics = compute_metrics(pbe_results)

print(f'\n===== Paint-by-Example Test Metrics =====')
print(f'  SSIM: {pbe_metrics["ssim_mean"]:.4f} +/- {pbe_metrics["ssim_std"]:.4f}')
print(f'  PSNR: {pbe_metrics["psnr_mean"]:.2f} +/- {pbe_metrics["psnr_std"]:.2f} dB')
print(f'=========================================')

In [ ]:
# Save generated images for FID computation
fid_real_dir = os.path.join(SAMPLE_DIR, 'fid_real_pbe')
fid_fake_dir = os.path.join(SAMPLE_DIR, 'fid_fake_pbe')
os.makedirs(fid_real_dir, exist_ok=True)
os.makedirs(fid_fake_dir, exist_ok=True)

for i, r in enumerate(pbe_results):
    r['target'].resize((256, 256)).save(os.path.join(fid_real_dir, f'{i:05d}.png'))
    r['generated'].resize((256, 256)).save(os.path.join(fid_fake_dir, f'{i:05d}.png'))

!python -m pytorch_fid {fid_real_dir} {fid_fake_dir}

In [ ]:
# Free GPU memory before loading next pipeline
del pipe_pbe
torch.cuda.empty_cache()
print('Paint-by-Example pipeline unloaded.')

## Approach 2: SD Inpainting + IP-Adapter (Zero-Shot)

This approach uses Stable Diffusion Inpainting with an IP-Adapter to condition on the furniture reference image.  
IP-Adapter adds a separate cross-attention path for image features, allowing "image prompts" alongside text.

**Conceptually similar to virtual try-on:** instead of placing clothing on a person, we place furniture in a room.

In [ ]:
from diffusers import StableDiffusionInpaintPipeline
from huggingface_hub import login

# Authenticate if needed (the inpainting model may be gated)
# Set your token here or run `huggingface-cli login` beforehand
HF_TOKEN = None  # e.g. 'hf_...'  or leave None if already logged in
if HF_TOKEN:
    login(token=HF_TOKEN)

pipe_ipa = StableDiffusionInpaintPipeline.from_pretrained(
    'runwayml/stable-diffusion-inpainting',
    torch_dtype=torch.float16,
    safety_checker=None,
    token=HF_TOKEN,
).to('cuda')

pipe_ipa.load_ip_adapter(
    'h94/IP-Adapter',
    subfolder='models',
    weight_name='ip-adapter_sd15.bin',
)
pipe_ipa.set_ip_adapter_scale(0.9)
pipe_ipa.set_progress_bar_config(disable=True)
print('SD Inpainting + IP-Adapter pipeline loaded.')

In [ ]:
@torch.inference_mode()
def run_ip_adapter_inpaint(pipe, split_df, num_samples=None, seed=42):
    """Run SD Inpainting + IP-Adapter on a split."""
    generator = torch.Generator('cuda').manual_seed(seed)
    results = []
    n = len(split_df) if num_samples is None else min(num_samples, len(split_df))

    for i in tqdm(range(n), desc='IP-Adapter Inpainting'):
        row = split_df.iloc[i]
        room = load_and_resize(row['input_path'])
        target = load_and_resize(row['target_path'])
        furn = load_and_resize(row['furniture_path'])
        mask = make_bbox_mask(row)

        output = pipe(
            prompt='a piece of furniture placed naturally in the room, photorealistic interior design',
            negative_prompt='blurry, distorted, low quality, unrealistic',
            image=room,
            mask_image=mask,
            ip_adapter_image=furn,
            num_inference_steps=NUM_STEPS,
            guidance_scale=7.5,
            strength=0.99,
            generator=generator,
        ).images[0]

        results.append({
            'room': room,
            'furniture': furn,
            'generated': output,
            'target': target,
            'mask': mask,
        })

    return results


# Preview on 8 samples
ipa_preview = run_ip_adapter_inpaint(pipe_ipa, test_df, num_samples=8)
show_results(ipa_preview, 'SD Inpainting + IP-Adapter Zero-Shot')

In [ ]:
# Full test set evaluation
print('Running IP-Adapter Inpainting on full test set...')
ipa_results = run_ip_adapter_inpaint(pipe_ipa, test_df)
ipa_metrics = compute_metrics(ipa_results)

print(f'\n===== SD Inpainting + IP-Adapter Test Metrics =====')
print(f'  SSIM: {ipa_metrics["ssim_mean"]:.4f} +/- {ipa_metrics["ssim_std"]:.4f}')
print(f'  PSNR: {ipa_metrics["psnr_mean"]:.2f} +/- {ipa_metrics["psnr_std"]:.2f} dB')
print(f'===================================================')

# FID
fid_real_dir_ipa = os.path.join(SAMPLE_DIR, 'fid_real_ipa')
fid_fake_dir_ipa = os.path.join(SAMPLE_DIR, 'fid_fake_ipa')
os.makedirs(fid_real_dir_ipa, exist_ok=True)
os.makedirs(fid_fake_dir_ipa, exist_ok=True)

for i, r in enumerate(ipa_results):
    r['target'].resize((256, 256)).save(os.path.join(fid_real_dir_ipa, f'{i:05d}.png'))
    r['generated'].resize((256, 256)).save(os.path.join(fid_fake_dir_ipa, f'{i:05d}.png'))

!python -m pytorch_fid {fid_real_dir_ipa} {fid_fake_dir_ipa}

In [ ]:
# Compare both approaches side by side
print('\n' + '=' * 55)
print('         ZERO-SHOT COMPARISON')
print('=' * 55)
print(f'{"Metric":<12} {"Paint-by-Example":>20} {"IP-Adapter":>20}')
print('-' * 55)
print(f'{"SSIM":<12} {pbe_metrics["ssim_mean"]:>17.4f}    {ipa_metrics["ssim_mean"]:>17.4f}')
print(f'{"PSNR (dB)":<12} {pbe_metrics["psnr_mean"]:>17.2f}    {ipa_metrics["psnr_mean"]:>17.2f}')
print('=' * 55)

In [ ]:
del pipe_ipa
torch.cuda.empty_cache()
print('IP-Adapter pipeline unloaded.')

## Fine-Tuning: Paint-by-Example for Border Blending

Fine-tune the Paint-by-Example U-Net to **blend furniture edges** — not generate furniture from scratch.

**Training data construction (paste-then-blend):**
- Take the empty room and paste the actual furniture crop at the bounding box → **composite**
- Create a **border ring mask** around the furniture edges (same as inference)
- Target is the ground truth furnished room
- The model learns to fix the seam, add shadows, and harmonize lighting at the boundary

**Training setup:**
- Loss: MSE denoising loss (standard diffusion training objective)
- Optimizer: AdamW with cosine LR schedule
- Mixed precision (fp16) + gradient checkpointing for A100 efficiency
- Checkpoint saving to Google Drive

In [ ]:
from diffusers import PaintByExamplePipeline, DDPMScheduler, AutoencoderKL, UNet2DConditionModel
from transformers import CLIPVisionModelWithProjection, CLIPImageProcessor
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

MODEL_ID = 'Fantasy-Studio/Paint-by-Example'
CHECKPOINT_DIR = '/content/drive/MyDrive/693-project/checkpoints/diffusion_pbe'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Load model components
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder='scheduler')
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder='vae', torch_dtype=torch.float16).to('cuda')
image_encoder = CLIPVisionModelWithProjection.from_pretrained(
    MODEL_ID, subfolder='image_encoder', torch_dtype=torch.float16
).to('cuda')
feature_extractor = CLIPImageProcessor.from_pretrained(MODEL_ID, subfolder='feature_extractor')
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder='unet', torch_dtype=torch.float16).to('cuda')

vae.requires_grad_(False)
image_encoder.requires_grad_(False)
unet.requires_grad_(True)
unet.enable_gradient_checkpointing()

trainable_params = sum(p.numel() for p in unet.parameters() if p.requires_grad)
print(f'Trainable UNet parameters: {trainable_params:,}')
print('VAE and Image Encoder frozen.')

In [ ]:
FT_BORDER_OUTER = 20
FT_BORDER_INNER = 6


def _make_border_mask(x1, y1, x2, y2, img_size, border_outer, border_inner):
    """Build a border-ring mask matching the inference pipeline."""
    from PIL import ImageDraw, ImageFilter

    ox1 = max(0, x1 - border_outer)
    oy1 = max(0, y1 - border_outer)
    ox2 = min(img_size, x2 + border_outer)
    oy2 = min(img_size, y2 + border_outer)

    ix1 = min(x1 + border_inner, x2)
    iy1 = min(y1 + border_inner, y2)
    ix2 = max(x2 - border_inner, x1)
    iy2 = max(y2 - border_inner, y1)

    mask = Image.new('L', (img_size, img_size), 0)
    draw = ImageDraw.Draw(mask)
    draw.rectangle([ox1, oy1, ox2, oy2], fill=255)
    draw.rectangle([ix1, iy1, ix2, iy2], fill=0)
    return mask.filter(ImageFilter.GaussianBlur(radius=4))


class FurniturePlacementDataset(Dataset):
    """Dataset for fine-tuning Paint-by-Example on border blending.

    Each sample constructs a composite (furniture pasted onto empty room)
    and a border-ring mask, matching the inference pipeline exactly.
    """

    def __init__(self, split_df, img_size=512, stored_size=256):
        self.df = split_df.reset_index(drop=True)
        self.img_size = img_size
        self.stored_size = stored_size
        self.to_tensor = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        room = Image.open(row['input_path']).convert('RGB').resize(
            (self.img_size, self.img_size), Image.BILINEAR)
        target = Image.open(row['target_path']).convert('RGB')
        furn = Image.open(row['furniture_path']).convert('RGB')

        scale = self.img_size / self.stored_size
        margin = 10
        x1 = max(0, int(row['bbox_x1'] * scale) - margin)
        y1 = max(0, int(row['bbox_y1'] * scale) - margin)
        x2 = min(self.img_size, int(row['bbox_x2'] * scale) + margin)
        y2 = min(self.img_size, int(row['bbox_y2'] * scale) + margin)

        # Paste the actual furniture crop onto the room → composite
        furn_paste = furn.resize((x2 - x1, y2 - y1), Image.LANCZOS)
        composite = room.copy()
        composite.paste(furn_paste, (x1, y1))

        # Border-ring mask (same geometry as inference)
        border_mask = _make_border_mask(
            x1, y1, x2, y2, self.img_size, FT_BORDER_OUTER, FT_BORDER_INNER)
        mask_tensor = transforms.ToTensor()(border_mask)

        # The composite with furniture-interior masked out
        composite_tensor = self.to_tensor(composite)
        masked_composite = composite_tensor * (1.0 - mask_tensor)

        target_tensor = self.to_tensor(target)
        furn_resized = furn.resize((224, 224), Image.BILINEAR)

        return {
            'target': target_tensor,
            'masked_room': masked_composite,
            'mask': mask_tensor,
            'furniture_pil': furn_resized,
        }


train_dataset = FurniturePlacementDataset(train_df)
val_dataset = FurniturePlacementDataset(val_df)

print(f'Train dataset: {len(train_dataset)} samples')
print(f'Val dataset: {len(val_dataset)} samples')

sample = train_dataset[0]
print(f'Target shape: {sample["target"].shape}')
print(f'Masked composite shape: {sample["masked_room"].shape}')
print(f'Border mask shape: {sample["mask"].shape}')

# Visualize a few training samples
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
col_titles = ['Masked Composite (Input)', 'Border Mask', 'Target', 'Furniture Ref']
for i in range(3):
    s = train_dataset[i]
    axes[i, 0].imshow(s['masked_room'].permute(1, 2, 0) * 0.5 + 0.5)
    axes[i, 1].imshow(s['mask'][0], cmap='gray')
    axes[i, 2].imshow(s['target'].permute(1, 2, 0) * 0.5 + 0.5)
    axes[i, 3].imshow(s['furniture_pil'])
    for j in range(4):
        axes[i, j].axis('off')
        if i == 0:
            axes[i, j].set_title(col_titles[j], fontsize=13, fontweight='bold')
plt.suptitle('Fine-Tuning Data: Border Blending Task', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
BATCH_SIZE = 4
NUM_EPOCHS = 50
LR = 1e-5
SAVE_EVERY = 10
SAMPLE_EVERY = 5


def collate_fn(batch):
    return {
        'target': torch.stack([b['target'] for b in batch]),
        'masked_room': torch.stack([b['masked_room'] for b in batch]),
        'mask': torch.stack([b['mask'] for b in batch]),
        'furniture_pil': [b['furniture_pil'] for b in batch],
    }


train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, collate_fn=collate_fn, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, collate_fn=collate_fn)

optimizer = torch.optim.AdamW(unet.parameters(), lr=LR, weight_decay=1e-2)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS * len(train_loader))
scaler = torch.amp.GradScaler('cuda')

print(f'Batches per epoch: {len(train_loader)}')
print(f'Total training steps: {NUM_EPOCHS * len(train_loader)}')

In [ ]:
def encode_reference_images(pil_images):
    """Encode furniture reference images using CLIP image encoder."""
    inputs = feature_extractor(images=pil_images, return_tensors='pt')
    pixel_values = inputs.pixel_values.to('cuda', dtype=torch.float16)
    image_embeds = image_encoder(pixel_values).image_embeds
    image_embeds = image_embeds.unsqueeze(1)
    return image_embeds


history = {'train_loss': [], 'val_loss': []}

print(f'Training for {NUM_EPOCHS} epochs...')
print('=' * 60)

for epoch in range(1, NUM_EPOCHS + 1):
    unet.train()
    epoch_loss = 0.0
    num_batches = 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS}')
    for batch in pbar:
        target = batch['target'].to('cuda', dtype=torch.float16)
        masked_room = batch['masked_room'].to('cuda', dtype=torch.float16)
        mask = batch['mask'].to('cuda', dtype=torch.float16)

        with torch.no_grad():
            latents = vae.encode(target).latent_dist.sample() * vae.config.scaling_factor
            masked_latents = vae.encode(masked_room).latent_dist.sample() * vae.config.scaling_factor
            mask_latent = F.interpolate(mask, size=latents.shape[-2:])
            encoder_hidden_states = encode_reference_images(batch['furniture_pil'])

        noise = torch.randn_like(latents)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                                  (latents.shape[0],), device='cuda').long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        unet_input = torch.cat([noisy_latents, mask_latent, masked_latents], dim=1)

        with torch.amp.autocast('cuda'):
            noise_pred = unet(unet_input, timesteps, encoder_hidden_states).sample
            loss = F.mse_loss(noise_pred, noise)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        lr_scheduler.step()

        epoch_loss += loss.item()
        num_batches += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_train = epoch_loss / num_batches
    history['train_loss'].append(avg_train)

    # Validation
    unet.eval()
    val_loss = 0.0
    val_batches = 0
    with torch.no_grad():
        for batch in val_loader:
            target = batch['target'].to('cuda', dtype=torch.float16)
            masked_room = batch['masked_room'].to('cuda', dtype=torch.float16)
            mask = batch['mask'].to('cuda', dtype=torch.float16)

            latents = vae.encode(target).latent_dist.sample() * vae.config.scaling_factor
            masked_latents = vae.encode(masked_room).latent_dist.sample() * vae.config.scaling_factor
            mask_latent = F.interpolate(mask, size=latents.shape[-2:])
            encoder_hidden_states = encode_reference_images(batch['furniture_pil'])

            noise = torch.randn_like(latents)
            timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                                      (latents.shape[0],), device='cuda').long()
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
            unet_input = torch.cat([noisy_latents, mask_latent, masked_latents], dim=1)

            noise_pred = unet(unet_input, timesteps, encoder_hidden_states).sample
            val_loss += F.mse_loss(noise_pred, noise).item()
            val_batches += 1

    avg_val = val_loss / max(val_batches, 1)
    history['val_loss'].append(avg_val)

    print(f'Epoch {epoch} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f} | '
          f'LR: {lr_scheduler.get_last_lr()[0]:.2e}')

    if epoch % SAVE_EVERY == 0:
        ckpt_path = os.path.join(CHECKPOINT_DIR, f'unet_epoch_{epoch:04d}')
        unet.save_pretrained(ckpt_path)
        print(f'  Checkpoint saved to {ckpt_path}')

# Save final checkpoint
final_path = os.path.join(CHECKPOINT_DIR, 'unet_final')
unet.save_pretrained(final_path)
print(f'\nTraining complete. Final checkpoint: {final_path}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('Denoising Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history['train_loss'], label='Train')
axes[1].plot(history['val_loss'], label='Val')
axes[1].set_title('Denoising Loss (log scale)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE Loss')
axes[1].set_yscale('log')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(SAMPLE_DIR, 'diffusion_loss_curves.png'), dpi=150)
plt.show()

## Evaluation: Fine-Tuned Model (Paste + Blend)

Load the fine-tuned UNet back into the Paint-by-Example pipeline and evaluate  
using the paste-then-blend approach (matching the training task).

In [ ]:
# Build pipeline with fine-tuned UNet
del optimizer, scaler, lr_scheduler
torch.cuda.empty_cache()

pipe_ft = PaintByExamplePipeline.from_pretrained(
    MODEL_ID,
    unet=unet,
    torch_dtype=torch.float16,
    safety_checker=None,
).to('cuda')
pipe_ft.set_progress_bar_config(disable=True)
print('Fine-tuned pipeline ready.')

In [ ]:
@torch.inference_mode()
def run_paste_and_blend(pipe, split_df, num_samples=None, seed=42):
    """Evaluate the fine-tuned model with paste-then-blend (matches training)."""
    from PIL import ImageDraw, ImageFilter

    generator = torch.Generator('cuda').manual_seed(seed)
    results = []
    n = len(split_df) if num_samples is None else min(num_samples, len(split_df))

    for i in tqdm(range(n), desc='Paste & Blend'):
        row = split_df.iloc[i]
        room = load_and_resize(row['input_path'])
        target = load_and_resize(row['target_path'])
        furn = load_and_resize(row['furniture_path'])

        scale = IMG_SIZE / 256
        margin = 10
        x1 = max(0, int(row['bbox_x1'] * scale) - margin)
        y1 = max(0, int(row['bbox_y1'] * scale) - margin)
        x2 = min(IMG_SIZE, int(row['bbox_x2'] * scale) + margin)
        y2 = min(IMG_SIZE, int(row['bbox_y2'] * scale) + margin)

        furn_paste = furn.resize((x2 - x1, y2 - y1), Image.LANCZOS)
        composite = room.copy()
        composite.paste(furn_paste, (x1, y1))

        border_mask = _make_border_mask(
            x1, y1, x2, y2, IMG_SIZE, FT_BORDER_OUTER, FT_BORDER_INNER)

        output = pipe(
            image=composite,
            mask_image=border_mask,
            example_image=furn,
            num_inference_steps=NUM_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            generator=generator,
        ).images[0]

        ix1 = min(x1 + FT_BORDER_INNER, x2)
        iy1 = min(y1 + FT_BORDER_INNER, y2)
        ix2 = max(x2 - FT_BORDER_INNER, x1)
        iy2 = max(y2 - FT_BORDER_INNER, y1)
        if ix2 > ix1 and iy2 > iy1:
            core = composite.crop((ix1, iy1, ix2, iy2))
            output.paste(core, (ix1, iy1))

        results.append({
            'room': room,
            'furniture': furn,
            'generated': output,
            'target': target,
            'mask': border_mask,
        })

    return results


print('Running fine-tuned Paste & Blend on full test set...')
ft_results = run_paste_and_blend(pipe_ft, test_df)
ft_metrics = compute_metrics(ft_results)

print(f'\n===== Fine-Tuned Paste & Blend Test Metrics =====')
print(f'  SSIM: {ft_metrics["ssim_mean"]:.4f} +/- {ft_metrics["ssim_std"]:.4f}')
print(f'  PSNR: {ft_metrics["psnr_mean"]:.2f} +/- {ft_metrics["psnr_std"]:.2f} dB')
print(f'=================================================')

show_results(ft_results, 'Fine-Tuned Paste and Blend')

In [ ]:
# FID for fine-tuned model
fid_real_ft = os.path.join(SAMPLE_DIR, 'fid_real_ft')
fid_fake_ft = os.path.join(SAMPLE_DIR, 'fid_fake_ft')
os.makedirs(fid_real_ft, exist_ok=True)
os.makedirs(fid_fake_ft, exist_ok=True)

for i, r in enumerate(ft_results):
    r['target'].resize((256, 256)).save(os.path.join(fid_real_ft, f'{i:05d}.png'))
    r['generated'].resize((256, 256)).save(os.path.join(fid_fake_ft, f'{i:05d}.png'))

!python -m pytorch_fid {fid_real_ft} {fid_fake_ft}

In [ ]:
# Final comparison across all approaches
print('\n' + '=' * 75)
print('              FINAL COMPARISON -- ALL APPROACHES')
print('=' * 75)
print(f'{"Metric":<12} {"PbE Zero-Shot":>18} {"IP-Adapter":>18} {"Paste+Blend FT":>18}')
print('-' * 75)
print(f'{"SSIM":<12} {pbe_metrics["ssim_mean"]:>15.4f}    {ipa_metrics["ssim_mean"]:>15.4f}    {ft_metrics["ssim_mean"]:>15.4f}')
print(f'{"PSNR (dB)":<12} {pbe_metrics["psnr_mean"]:>15.2f}    {ipa_metrics["psnr_mean"]:>15.2f}    {ft_metrics["psnr_mean"]:>15.2f}')
print('=' * 75)
print()
print('PbE Zero-Shot / IP-Adapter: generate furniture from scratch (full bbox)')
print('Paste+Blend FT: paste real crop, diffusion blends edges only')

In [ ]:
# Side-by-side comparison: best results from each approach
n = min(6, len(test_df))
fig, axes = plt.subplots(n, 5, figsize=(25, 5 * n))
col_titles = ['Empty Room', 'Furniture Ref', 'PbE Zero-Shot', 'Paste+Blend FT', 'Ground Truth']

for i in range(n):
    axes[i, 0].imshow(pbe_results[i]['room'])
    axes[i, 1].imshow(pbe_results[i]['furniture'])
    axes[i, 2].imshow(pbe_results[i]['generated'])
    axes[i, 3].imshow(ft_results[i]['generated'])
    axes[i, 4].imshow(pbe_results[i]['target'])
    for j in range(5):
        axes[i, j].axis('off')
        if i == 0:
            axes[i, j].set_title(col_titles[j], fontsize=13, fontweight='bold')

plt.suptitle('Diffusion Furniture Staging — Final Comparison',
             fontsize=18, fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig(os.path.join(SAMPLE_DIR, 'final_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

---

## End-to-End Inference: Segment → Place → Paste → Blend

The full virtual home staging pipeline uses the **actual furniture crop** (not a regenerated version):

1. **Mask2Former** (pre-trained on ADE20K) segments the room into floor, walls, windows, doors
2. **Placement heuristics** find a valid furniture location from the floor/wall layout
3. **Paste** the real furniture crop (resized) onto the room at that location
4. **Paint-by-Example** inpaints only a **thin border ring** around the furniture edges to blend seams and generate natural shadows

The furniture pixels themselves are preserved exactly — diffusion only touches the boundary region.

In [ ]:
from transformers import Mask2FormerForUniversalSegmentation, Mask2FormerImageProcessor

m2f_processor = Mask2FormerImageProcessor.from_pretrained(
    'facebook/mask2former-swin-large-ade-semantic'
)
m2f_model = Mask2FormerForUniversalSegmentation.from_pretrained(
    'facebook/mask2former-swin-large-ade-semantic'
)
m2f_model.eval()
if torch.cuda.is_available():
    m2f_model = m2f_model.cuda()
print('Mask2Former loaded (ADE20K semantic segmentation).')

In [ ]:
ADE20K_FLOOR = 3
ADE20K_WALL  = 0


def segment_room(image_pil):
    """Run Mask2Former semantic segmentation. Returns a 2D array of class IDs."""
    inputs = m2f_processor(images=image_pil, return_tensors='pt')
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    with torch.no_grad():
        outputs = m2f_model(**inputs)

    pred = m2f_processor.post_process_semantic_segmentation(
        outputs, target_sizes=[image_pil.size[::-1]]
    )[0]
    return pred.cpu().numpy()


def find_placement_bbox(seg_map, furniture_type='bed'):
    """Find a plausible placement location based on room segmentation.

    Strategy: find floor pixels adjacent to walls, place furniture there
    with size proportional to the room.
    """
    floor_mask = (seg_map == ADE20K_FLOOR).astype(np.uint8)
    wall_mask  = (seg_map == ADE20K_WALL).astype(np.uint8)
    h, w = seg_map.shape

    if floor_mask.sum() < 100:
        fw, fh = int(w * 0.4), int(h * 0.3)
        cx, cy = w // 2, int(h * 0.65)
        return (cx - fw // 2, cy - fh // 2, cx + fw // 2, cy + fh // 2)

    floor_rows = np.where(floor_mask.sum(axis=1) > w * 0.1)[0]
    if len(floor_rows) == 0:
        floor_rows = np.array([int(h * 0.5), int(h * 0.8)])

    floor_top = floor_rows.min()
    floor_bot = floor_rows.max()

    if furniture_type == 'bed':
        fw = int(w * 0.35)
        fh = int((floor_bot - floor_top) * 0.5)
    elif furniture_type in ('sofa', 'couch'):
        fw = int(w * 0.4)
        fh = int((floor_bot - floor_top) * 0.35)
    else:
        fw = int(w * 0.25)
        fh = int((floor_bot - floor_top) * 0.3)

    fw = max(fw, 30)
    fh = max(fh, 30)

    wall_cols = np.where(wall_mask.sum(axis=0) > h * 0.2)[0]
    if len(wall_cols) > 0:
        wall_center = int(np.median(wall_cols))
        cx = np.clip(wall_center, fw // 2, w - fw // 2)
    else:
        cx = w // 2

    cy = int(floor_top + (floor_bot - floor_top) * 0.4)
    cy = np.clip(cy, fh // 2, h - fh // 2)

    x1 = max(0, cx - fw // 2)
    y1 = max(0, cy - fh // 2)
    x2 = min(w, x1 + fw)
    y2 = min(h, y1 + fh)

    return (x1, y1, x2, y2)

In [ ]:
BORDER_OUTER = 20   # pixels of room around the bbox to inpaint (shadows, reflections)
BORDER_INNER = 6    # pixels into the furniture edge to inpaint (seam blending)


@torch.inference_mode()
def end_to_end_staging(pipe, room_pil, furniture_pil, furniture_type='bed',
                       num_steps=50, guidance_scale=5.0, seed=42):
    """Full end-to-end pipeline: segment → place → paste → blend edges.

    The actual furniture crop is pasted directly. Diffusion only inpaints a
    thin border ring around the furniture to blend seams and add shadows.
    """
    from PIL import ImageDraw, ImageFilter

    img_size = 512
    room_resized = room_pil.resize((img_size, img_size), Image.BILINEAR)

    seg_map = segment_room(room_resized)
    bbox = find_placement_bbox(seg_map, furniture_type)
    x1, y1, x2, y2 = bbox

    # --- Paste the actual furniture crop onto the room ---
    furn_resized = furniture_pil.convert('RGB').resize(
        (x2 - x1, y2 - y1), Image.LANCZOS
    )
    composite = room_resized.copy()
    composite.paste(furn_resized, (x1, y1))

    # --- Build a border-ring mask (only the seam area, not the furniture interior) ---
    outer_x1 = max(0, x1 - BORDER_OUTER)
    outer_y1 = max(0, y1 - BORDER_OUTER)
    outer_x2 = min(img_size, x2 + BORDER_OUTER)
    outer_y2 = min(img_size, y2 + BORDER_OUTER)

    inner_x1 = min(x1 + BORDER_INNER, x2)
    inner_y1 = min(y1 + BORDER_INNER, y2)
    inner_x2 = max(x2 - BORDER_INNER, x1)
    inner_y2 = max(y2 - BORDER_INNER, y1)

    border_mask = Image.new('L', (img_size, img_size), 0)
    draw = ImageDraw.Draw(border_mask)
    draw.rectangle([outer_x1, outer_y1, outer_x2, outer_y2], fill=255)
    draw.rectangle([inner_x1, inner_y1, inner_x2, inner_y2], fill=0)

    # Gaussian-blur the mask for a softer transition
    border_mask = border_mask.filter(ImageFilter.GaussianBlur(radius=4))

    # --- Diffusion inpaints only the border ring ---
    generator = torch.Generator('cuda').manual_seed(seed)
    blended = pipe(
        image=composite,
        mask_image=border_mask,
        example_image=furniture_pil,
        num_inference_steps=num_steps,
        guidance_scale=guidance_scale,
        generator=generator,
    ).images[0]

    # --- Guarantee furniture-core pixels are untouched ---
    # Paste the original furniture interior back (excluding the thin BORDER_INNER strip)
    if inner_x2 > inner_x1 and inner_y2 > inner_y1:
        core_crop = composite.crop((inner_x1, inner_y1, inner_x2, inner_y2))
        blended.paste(core_crop, (inner_x1, inner_y1))

    return {
        'room': room_resized,
        'segmentation': seg_map,
        'bbox': bbox,
        'mask': border_mask,
        'composite': composite,
        'staged': blended,
        'furniture': furniture_pil,
    }

In [ ]:
def visualize_segmentation(seg_map, ax=None, alpha=0.6):
    """Color-code the segmentation map for visualization."""
    h, w = seg_map.shape
    unique_classes = np.unique(seg_map)
    rng = np.random.RandomState(42)
    color_map = rng.randint(50, 240, (200, 3), dtype=np.uint8)
    color_map[ADE20K_FLOOR] = [180, 180, 220]
    color_map[ADE20K_WALL]  = [220, 200, 180]

    vis = np.zeros((h, w, 3), dtype=np.uint8)
    for c in unique_classes:
        vis[seg_map == c] = color_map[c % len(color_map)]

    if ax is not None:
        ax.imshow(vis)
    return vis

### End-to-End Demo on Test Samples

Run the full pipeline: **Room → Segmentation → Placement → Diffusion → Staged Room**

We also compare against the ground truth (which used the original bounding box from the dataset).

In [ ]:
# Use the fine-tuned pipeline (pipe_ft) for end-to-end demo
# If not available, fall back to loading it
try:
    pipe_ft
    print('Using fine-tuned pipeline (already in memory).')
except NameError:
    print('Loading fine-tuned pipeline from checkpoint...')
    ft_unet = UNet2DConditionModel.from_pretrained(
        os.path.join(CHECKPOINT_DIR, 'unet_final'),
        torch_dtype=torch.float16,
    )
    pipe_ft = PaintByExamplePipeline.from_pretrained(
        MODEL_ID,
        unet=ft_unet,
        torch_dtype=torch.float16,
        safety_checker=None,
    ).to('cuda')
    pipe_ft.set_progress_bar_config(disable=True)
    print('Fine-tuned pipeline loaded from checkpoint.')

NUM_E2E_SAMPLES = 8

print(f'\nRunning end-to-end pipeline on {NUM_E2E_SAMPLES} test samples...\n')
e2e_results = []
for i in tqdm(range(NUM_E2E_SAMPLES), desc='End-to-End Staging'):
    row = test_df.iloc[i]
    room_pil = load_and_resize(row['input_path'])
    furn_pil = load_and_resize(row['furniture_path'])
    target_pil = load_and_resize(row['target_path'])

    result = end_to_end_staging(pipe_ft, room_pil, furn_pil, furniture_type='bed')
    result['target'] = target_pil
    e2e_results.append(result)

print('Done.')

In [ ]:
n = len(e2e_results)
fig, axes = plt.subplots(n, 7, figsize=(35, 5 * n))
col_titles = ['Empty Room', 'Segmentation', 'Border Mask', 'Furniture Ref',
              'Composite (Pasted)', 'Blended (Final)', 'Ground Truth']

for i, r in enumerate(e2e_results):
    axes[i, 0].imshow(r['room'])
    visualize_segmentation(r['segmentation'], ax=axes[i, 1])
    axes[i, 2].imshow(r['mask'], cmap='gray')
    axes[i, 3].imshow(r['furniture'])
    axes[i, 4].imshow(r['composite'])
    axes[i, 5].imshow(r['staged'])
    axes[i, 6].imshow(r['target'])

    for j in range(7):
        axes[i, j].axis('off')
        if i == 0:
            axes[i, j].set_title(col_titles[j], fontsize=13, fontweight='bold')

plt.suptitle('End-to-End: Segment → Place → Paste → Blend',
             fontsize=18, fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig(os.path.join(SAMPLE_DIR, 'end_to_end_results.png'),
            dpi=150, bbox_inches='tight')
plt.show()

### End-to-End Quantitative Evaluation

Run the full pipeline on the entire test set and compute metrics.

**Important note:** Metrics like SSIM/PSNR compare against the ground truth, which used a *different* bounding box (from the dataset). Since our pipeline places furniture heuristically, the placement location will differ — so absolute metrics will be lower than the oracle-bbox evaluation above. What matters is that the **generated furniture looks realistic and spatially coherent**.

In [ ]:
print('Running full end-to-end evaluation on test set...')
e2e_full_results = []

for i in tqdm(range(len(test_df)), desc='E2E Full Test'):
    row = test_df.iloc[i]
    room_pil = load_and_resize(row['input_path'])
    furn_pil = load_and_resize(row['furniture_path'])
    target_pil = load_and_resize(row['target_path'])

    result = end_to_end_staging(pipe_ft, room_pil, furn_pil, furniture_type='bed')

    e2e_full_results.append({
        'room': room_pil,
        'furniture': furn_pil,
        'generated': result['staged'],
        'target': target_pil,
        'mask': result['mask'],
    })

e2e_metrics = compute_metrics(e2e_full_results)

print(f'\n{"=" * 60}')
print(f'  END-TO-END PIPELINE — Test Set Metrics')
print(f'  (placement is automatic, NOT oracle bounding box)')
print(f'{"=" * 60}')
print(f'  SSIM: {e2e_metrics["ssim_mean"]:.4f} +/- {e2e_metrics["ssim_std"]:.4f}')
print(f'  PSNR: {e2e_metrics["psnr_mean"]:.2f} +/- {e2e_metrics["psnr_std"]:.2f} dB')
print(f'{"=" * 60}')

# FID for end-to-end
fid_real_e2e = os.path.join(SAMPLE_DIR, 'fid_real_e2e')
fid_fake_e2e = os.path.join(SAMPLE_DIR, 'fid_fake_e2e')
os.makedirs(fid_real_e2e, exist_ok=True)
os.makedirs(fid_fake_e2e, exist_ok=True)

for i, r in enumerate(e2e_full_results):
    r['target'].resize((256, 256)).save(os.path.join(fid_real_e2e, f'{i:05d}.png'))
    r['generated'].resize((256, 256)).save(os.path.join(fid_fake_e2e, f'{i:05d}.png'))

!python -m pytorch_fid {fid_real_e2e} {fid_fake_e2e}

In [ ]:
print('\n' + '=' * 85)
print('                COMPLETE COMPARISON — ALL APPROACHES')
print('=' * 85)
print(f'{"Approach":<28} {"Placement":<18} {"Furniture":<14} {"SSIM":>8} {"PSNR":>8}')
print('-' * 85)
print(f'{"PbE Zero-Shot":<28} {"Oracle bbox":<18} {"Generated":<14} '
      f'{pbe_metrics["ssim_mean"]:>8.4f} {pbe_metrics["psnr_mean"]:>8.2f}')
print(f'{"IP-Adapter Zero-Shot":<28} {"Oracle bbox":<18} {"Generated":<14} '
      f'{ipa_metrics["ssim_mean"]:>8.4f} {ipa_metrics["psnr_mean"]:>8.2f}')
print(f'{"Paste+Blend FT":<28} {"Oracle bbox":<18} {"Exact crop":<14} '
      f'{ft_metrics["ssim_mean"]:>8.4f} {ft_metrics["psnr_mean"]:>8.2f}')
print(f'{"Paste+Blend FT (E2E)":<28} {"Mask2Former":<18} {"Exact crop":<14} '
      f'{e2e_metrics["ssim_mean"]:>8.4f} {e2e_metrics["psnr_mean"]:>8.2f}')
print('=' * 85)
print()
print('Generated  = diffusion creates new furniture pixels (inspired by reference)')
print('Exact crop = real furniture pixels preserved, diffusion blends edges only')
print('E2E uses Mask2Former auto-placement → lower metrics vs ground truth expected.')

---

### Try with Your Own Images

Upload an empty room photo and a furniture reference image to run the full pipeline.

In [ ]:
# Upload your own images (run in Colab)
from google.colab import files as colab_files

print('Upload an EMPTY ROOM image:')
uploaded_room = colab_files.upload()
room_name = list(uploaded_room.keys())[0]
custom_room = Image.open(room_name).convert('RGB')

print('\nUpload a FURNITURE REFERENCE image:')
uploaded_furn = colab_files.upload()
furn_name = list(uploaded_furn.keys())[0]
custom_furn = Image.open(furn_name).convert('RGB')

custom_result = end_to_end_staging(
    pipe_ft, custom_room, custom_furn,
    furniture_type='bed', num_steps=50, guidance_scale=5.0
)

fig, axes = plt.subplots(1, 6, figsize=(30, 5))
axes[0].imshow(custom_result['room']); axes[0].set_title('Empty Room')
visualize_segmentation(custom_result['segmentation'], ax=axes[1]); axes[1].set_title('Segmentation')
axes[2].imshow(custom_result['mask'], cmap='gray'); axes[2].set_title('Border Mask')
axes[3].imshow(custom_result['composite']); axes[3].set_title('Composite (Pasted)')
axes[4].imshow(custom_result['staged']); axes[4].set_title('Blended (Final)')
axes[5].imshow(custom_result['furniture']); axes[5].set_title('Furniture Ref')

for ax in axes:
    ax.axis('off')

plt.suptitle('Custom Image — Paste & Blend Virtual Home Staging',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()